# Visualizing ghost molecules

<video controls src="./assets/ghost_molecules.webm">


Load the nanotube + methane example recording as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("../recordings/nanotube-example-recording.nanover.zip")

Define a structure to hold data for ghost molecules:

In [2]:
import MDAnalysis as mda
import numpy.typing as npt
from dataclasses import dataclass

@dataclass(kw_only=True)
class GhostMolecule:
    atom_positions: npt.NDArray
    bond_pairs: list[tuple[int, int]]

    @classmethod
    def from_atom_group(cls, group: mda.AtomGroup):
        universe = mda.Merge(group)
        return cls(
            atom_positions=universe.atoms.positions / 10,  # angstrom -> nm
            bond_pairs=universe.bonds.indices,
        )


Track the relative position of the nanotube in each frame:

In [3]:
from nanover.utilities.transforms import find_transformation_between_point_patterns, Transform

NANOTUBE_ATOMS = list(range(0, 60))

_ = universe.trajectory[0]
NANOTUBE_INITIAL_POSITIONS = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm

NANOTUBE_TRANSFORMS: list[Transform] = []
for _ in universe.trajectory:
    nanotube_positions = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm
    matrix = find_transformation_between_point_patterns(NANOTUBE_INITIAL_POSITIONS, nanotube_positions)
    transform = Transform.from_parent_to_local_matrix(matrix)
    NANOTUBE_TRANSFORMS.append(transform)

Define a function to extract ghost molecules from particular trajectory frames and translate them into a common reference frame:

In [4]:
METHANE_ATOMS = list(range(60, 65))

GHOST_MOLECULES: list[GhostMolecule] = []
def set_ghost_frames(*indexes):
    GHOST_MOLECULES.clear()
    for index in indexes:
        transform_i = NANOTUBE_TRANSFORMS[index]
        _ = universe.trajectory[index]
        ghost = GhostMolecule.from_atom_group(universe.atoms[METHANE_ATOMS])
        ghost.atom_positions = transform_i.points_local_to_parent(ghost.atom_positions)
        GHOST_MOLECULES.append(ghost)

Choose which frames to visualise as ghosts:

In [5]:
set_ghost_frames(500, 610, 775, len(universe.trajectory)-1)

Set up the server with playback of the universe:

In [6]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: trajectory seeker")
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [7]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Define a FrameListener to constantly update the ghost visuals as the nanotube moves during playback:

In [8]:
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener

class RelativeGhostVisuals(FrameListener):
    points = []

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        frame_i_transform = NANOTUBE_TRANSFORMS[simulation.prev_frame]

        for s, ghost in enumerate(GHOST_MOLECULES):
            for i, position in enumerate(ghost.atom_positions):
                utilities.objects.update_shape(f"ghost.{s}.{i}", position=frame_i_transform.point_parent_to_local(position), size=0.1, color=[1.0, 1.0, 1.0, 0.25])
            for i, (a, b) in enumerate(ghost.bond_pairs):
                positions = frame_i_transform.points_parent_to_local(ghost.atom_positions[[a, b]])
                utilities.objects.update_line(f"ghost.{s}.{i}", positions=positions, size=0.075, color=[1.0, 1.0, 1.0, 0.5])

visuals = RelativeGhostVisuals.from_runner(imd_runner)
visuals.start()

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt
